<a href="https://colab.research.google.com/github/MSoledadGarcia/dm2025b/blob/main/src/Prueba%2320/Copia_de_boruta_prueba.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# primero establecer el Runtime de Python 3
from google.colab import drive
drive.mount('/content/.drive')

Mounted at /content/.drive


In [ ]:
%%shell

mkdir -p "/content/.drive/My Drive/dm"
mkdir -p "/content/buckets"
ln -s "/content/.drive/My Drive/dm" /content/buckets/b1

mkdir -p ~/.kaggle
cp /content/buckets/b1/kaggle/kaggle.json  ~/.kaggle
chmod 600 ~/.kaggle/kaggle.json


mkdir -p /content/buckets/b1/exp
mkdir -p /content/buckets/b1/datasets
mkdir -p /content/datasets



archivo_origen="https://storage.googleapis.com/open-courses/itba2025-8d0a/dataset_pequeno.csv"
archivo_destino="/content/datasets/dataset_pequeno.csv"
archivo_destino_bucket="/content/buckets/b1/datasets/dataset_pequeno.csv"

if ! test -f $archivo_destino_bucket; then
  wget  $archivo_origen  -O $archivo_destino_bucket
fi


if ! test -f $archivo_destino; then
  cp  $archivo_destino_bucket  $archivo_destino
fi


Entorno de ejecucion R


In [ ]:
format(Sys.time(), "%a %b %d %X %Y")

[1] "Mon Sep 08 01:31:04 AM 2025"

In [ ]:
# limpio la memoria
rm(list=ls(all.names=TRUE)) # remove all objects
gc(full=TRUE, verbose=FALSE) # garbage collection

,used,(Mb),gc trigger,(Mb),max used,(Mb)
Ncells,659629,35.3,1454391,77.7,1454391,77.7
Vcells,1225685,9.4,8388608,64.0,1975128,15.1


In [ ]:
# cargo las librerias que necesito
require("data.table")
require("rpart")
if(!require("rpart.plot")) install.packages("rpart.plot")
require("rpart.plot")

   #cargo Boruta
if(!require("Boruta")) install.packages("Boruta")
require("Boruta")


Loading required package: data.table

Loading required package: rpart

Loading required package: rpart.plot

Warning message in library(package, lib.loc = lib.loc, character.only = TRUE, logical.return = TRUE, :
“there is no package called ‘rpart.plot’”
Installing package into ‘/usr/local/lib/R/site-library’
(as ‘lib’ is unspecified)

Loading required package: rpart.plot

Loading required package: Boruta

Warning message in library(package, lib.loc = lib.loc, character.only = TRUE, logical.return = TRUE, :
“there is no package called ‘Boruta’”
Installing package into ‘/usr/local/lib/R/site-library’
(as ‘lib’ is unspecified)

also installing the dependencies ‘RcppEigen’, ‘ranger’


Loading required package: Boruta



In [ ]:
PARAM <- list()
PARAM$experimento <- 001
PARAM$semilla_primigenia <- 100019

In [ ]:

# carpeta de trabajo
setwd("/content/buckets/b1/exp")
experimento_folder <- paste0("Boruta", PARAM$experimento)
dir.create(experimento_folder, showWarnings=FALSE)
setwd( paste0("/content/buckets/b1/exp/", experimento_folder ))

In [ ]:
# lectura del dataset
dataset <- fread("/content/datasets/dataset_pequeno.csv")

In [ ]:
# defino los dataset de entrenamiento y aplicacion
dtrain <- dataset[foto_mes == 202107]
dfuture <- dataset[foto_mes == 202109]

In [ ]:
#summary(dtrain)

Convierto clase ternaria a factor, ya que random forest necesita ese formato

In [ ]:
dtrain$clase_ternaria <- as.factor(dtrain$clase_ternaria)


In [ ]:

# Medir tiempo de ejecución de una corrida de Boruta
set.seed(123)

tiempo <- system.time({
  boruta.test <- Boruta(clase_ternaria ~ .,
                        data = dtrain,
                        doTrace = 1,   # log simple
                        ntree = 200,
                        maxRuns = 1)   # SOLO 1 corrida
})

print(tiempo)

#Me da el tiempo de una sola corrida, lo tengo q multiplicar x maxRuns

In [ ]:


# Nombre del archivo donde guardaremos el progreso
archivo_boruta <- "boruta_resultado.rds"

# Chequear si ya existe un resultado previo
if (file.exists(archivo_boruta)) {
  # Si existe, lo cargo
  message(" Cargando Boruta desde archivo guardado...")
  boruta.train <- readRDS(archivo_boruta)
} else {
  # Si no existe, corro Boruta desde cero
  message("Ejecutando Boruta desde cero...")
  set.seed(123)
  boruta.train <- Boruta(clase_ternaria ~ .,
                         data = dtrain,
                         doTrace = 2,
                         ntree = 200,
                         maxRuns = 20) #defaul=100, cambio para q sea mas rapido

  # Guardar el resultado para continuar más adelante si se corta
  saveRDS(boruta.train, archivo_boruta)
}


Ejecutando Boruta desde cero...

 1. run of importance source...



In [ ]:
#set.seed(123)
#boruta.train <- Boruta(clase_ternaria ~., data = dtrain, doTrace = 2)
#print(boruta.train)

In [ ]:
format(Sys.time(), "%a %b %d %X %Y")

In [ ]:
# Estado final de cada variable
boruta.train$finalDecision

# Resumen rápido
table(boruta.train$finalDecision)

# Variables confirmadas como importantes
getSelectedAttributes(boruta.train, withTentative = FALSE)

# Variables importantes + tentativas (por si querés reanalizar)
getSelectedAttributes(boruta.train, withTentative = TRUE)

# Información detallada
attStats(boruta.train)  # dataframe con medias, medianas y decisiones


In [ ]:
plot(boruta.train, las = 2, cex.axis = 0.7)


In [ ]:
#ver q hace
plot(boruta.train, xlab = "", xaxt = "n")
lz<-lapply(1:ncol(boruta.train$ImpHistory),function(i)
boruta.train$ImpHistory[is.finite(boruta.train$ImpHistory[,i]),i])
names(lz) <- colnames(boruta.train$ImpHistory)
Labels <- sort(sapply(lz,median))
axis(side = 1,las=2,labels = names(Labels),
at = 1:ncol(boruta.train$ImpHistory), cex.axis = 0.7)